# BIST Altin Bazli ATL / ATH Tarayici v6

Her hisse icin TradingView'deki `BIST:HISSE / FX_IDC:XAUTRY` grafiginin ATL ve ATH degerlerine uzaklik hesaplanir.

**v6 (veri duzeltmesi):**
- Veri artik dogrudan **TradingView**'den cekiliyor: grafikte gordugun ile ayni seri (1990'lara kadar).
  Eski surumler Yahoo'dan 2005 sonrasini aliyordu; hem eski dipleri goremiyordu hem de Yahoo'nun
  bedelsiz / temettu duzeltmeleri hataliydi.
  Ornek: ENKAI ATL = 0,00008718 (18.09.1998), TradingView grafigiyle birebir ayni.
- ATL = en dusuk fitil, ATH = en yuksek fitil (TradingView'deki ATL etiketi gibi). Ayarlardan kapanisa cevrilebilir.
- Tek gunluk hatali mumlar ayiklanir (ornek: BFREN 30.09.2004).
- Temettu duzeltmesi acik (TradingView'deki "Temettuye gore duzelt" ayari). Ayarlardan kapatilabilir.
- Tek hisse kontrol hucresi: bir hissenin dip / tepe gunlerini gosterir.

**Calistir:** Calisma zamani -> Tumunu calistir

In [ ]:
#@title Ayarlar
ESIK_YUZDE = 30  #@param {type:"slider", min:5, max:100, step:5}
#@markdown Radar esigi (%): ATL'ye bu kadar yakin olanlar isaretlenir
FIYAT = 'fitil'  #@param ["fitil", "kapanis"]
#@markdown fitil: ATL = en dusuk fiyat, ATH = en yuksek fiyat. kapanis: gunluk kapanislar
TEMETTU_DUZELTME = True  #@param {type:"boolean"}
#@markdown TradingView'deki "Temettuye gore duzelt" ayari ile ayni olmali
MIN_VERI_GUNU = 30  #@param {type:"number"}
ES_ZAMANLI = 6  #@param {type:"number"}
#@markdown Ayni anda kac hisse indirilsin (cok yukseltme, TradingView engelleyebilir)
ALTIN = 'FX_IDC:XAUTRY'  #@param {type:"string"}

In [ ]:
#@title Kurulum
!pip install -U websocket-client pandas requests -q
import pandas as pd, requests, time, re, json, random, string
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from websocket import create_connection
from IPython.display import display
print('Kurulum tamam')

In [ ]:
#@title Guncel BIST hisse listesini cek

UA = {'User-Agent': 'Mozilla/5.0'}
# Hisse olmayan / taramaya girmemesi gerekenler (ALTIN = Darphane altin sertifikasi)
HARIC = {'ALTIN'}

def tradingview_listesi():
    """TradingView BIST tarayicisindan islem goren tum hisse senetleri."""
    payload = {
        'filter': [{'left': 'type', 'operation': 'equal', 'right': 'stock'},
                   {'left': 'exchange', 'operation': 'equal', 'right': 'BIST'}],
        'columns': ['name'],
        'range': [0, 3000],
    }
    r = requests.post('https://scanner.tradingview.com/turkey/scan',
                      json=payload, headers=UA, timeout=20)
    r.raise_for_status()
    return {x['d'][0].upper() for x in r.json()['data']
            if re.fullmatch(r'[A-Z0-9]{3,6}', x['d'][0].upper())}

def isyatirim_listesi():
    """Is Yatirim temel degerler sayfasindaki hisse kodlari."""
    url = 'https://www.isyatirim.com.tr/tr-tr/analiz/hisse/Sayfalar/Temel-Degerler-Ve-Oranlar.aspx'
    r = requests.get(url, headers=UA, timeout=30)
    r.raise_for_status()
    return set(re.findall(r'hisse=([A-Z0-9]{3,6})', r.text))

# Statik yedek liste (TradingView'den, Ekim 2026 - 625 hisse)
STATIK = set("""
A1CAP A1YEN AAGYO ACSEL ADEL ADESE ADGYO AEFES AFYON AGESA AGHOL AGROT AGYO AHGAZ AHSGY AKBNK
AKCNS AKENR AKFGY AKFIS AKFYE AKGRT AKHAN AKMGY AKSA AKSEN AKSGY AKSUE AKYHO ALARK ALBRK ALBTN
ALCAR ALCTL ALFAS ALGYO ALKA ALKIM ALKLC ALTNY ALVES ANELE ANGEN ANHYT ANSGR ARASE ARCLK ARDYZ
ARENA ARFYE ARMGD ARSAN ARTMS ARZUM ASELS ASGYO ASTOR ASUZU ATAGY ATAKP ATATP ATATR ATEKS ATLAS
ATSYH AVGYO AVHOL AVOD AVPGY AYCES AYDEM AYEN AYES AYGAZ AZTEK BAGFS BAHKM BAKAB BALAT BALSU
BANVT BARMA BASCM BASGZ BAYRK BEGYO BERA BESLR BESTE BETAE BEYAZ BFREN BIENY BIGCH BIGEN BIGTK
BIMAS BINBN BINHO BIOEN BIZIM BJKAS BKRGY BLCYT BLUME BMSCH BMSTL BNTAS BOBET BORLS BORSK BOSSA
BRISA BRKO BRKSN BRKVY BRLSM BRMEN BRSAN BRYAT BSOKE BTCIM BUCIM BULGS BURCE BURVA BVSAN BYDNR
CANTE CASA CATES CCOLA CELHA CEMAS CEMTS CEMZY CEOEM CGCAM CIMSA CITAS CLEBI CMBTN CMENT CONSE
COSMO CRDFA CRFSA CUSAN CVKMD CWENE DAGI DAPGM DARDL DCTTR DENGE DERHL DERIM DESA DESPC DEVA
DGATE DGGYO DGNMO DIRIT DITAS DMLKT DMRGD DMSAS DNISI DOAS DOCO DOFER DOFRB DOGUB DOHOL DOKTA
DSTKF DUNYH DURDO DURKN DYOBY DZGYO EBEBK ECILC ECOGR ECZYT EDATA EDIP EFOR EGEEN EGEGY EGEPO
EGGUB EGPRO EGSER EKDMR EKGYO EKIM EKIZ EKOS EKSUN ELITE EMKEL EMNIS EMPAE ENDAE ENERY ENJSA
ENKAI ENPRA ENSRI ENTRA EPLAS ERBOS ERCB EREGL ERSU ESCAR ESCOM ESEN ETILR ETYAT EUHOL EUKYO
EUPWR EUREN EUYO EYGYO FADE FENER FLAP FMIZP FONET FORMT FORTE FRIGO FRMPL FROTO FZLGY GARAN
GARFA GATEG GEDIK GEDZA GENIL GENKM GENTS GEREL GESAN GIPTA GLBMD GLCVY GLRMK GLRYH GLYHO GMTAS
GOKNR GOLDA GOLTS GOODY GOZDE GRNYO GRSEL GRTHO GSDDE GSDHO GSRAY GUBRF GUNDG GWIND GZNMI HALKB
HATEK HATSN HDFGS HEDEF HEKTS HKTM HLGYO HOROZ HRKET HTTBT HUBVC HUNER HURGZ ICBCT ICUGS IDGYO
IEYHO IHAAS IHEVA IHGZT IHLAS IHLGM IHYAY IMASM INDES INFO INGRM INTEK INTEM INTET INVEO INVES
ISBIR ISBTR ISCTR ISDMR ISFIN ISGYO ISKPL ISMEN ISSEN ISVEA IZENR IZFAS IZINV IZMDC JANTS KAPLM
KARCL KAREL KARSN KARTN KATMR KAYSE KBORU KCAER KCHOL KENT KERVN KFEIN KGYO KIMMR KLGYO KLKIM
KLMSN KLNMA KLRHO KLSER KLSYN KLYPV KMPUR KNFRT KOCMT KONKA KONTR KONYA KOPOL KORDS KOTON KPEKS
KRDMA KRDMB KRDMD KRGYO KRONT KRPLS KRSTL KRTEK KRVGD KSTUR KTLEV KTSKR KUTPO KUVVA KUYAS KZBGY
KZGYO LIDER LIDFA LILAK LINK LKMNH LMKDC LOGO LRSHO LUKSK LXGYO LYDHO LYDYE MAALT MACKO MAGEN
MAKIM MAKTK MANAS MARBL MARMR MARTI MASFN MAVI MCARD MEDTR MEGAP MEGMT MEKAG MEPET MERCN MERIT
MERKO METEN METRO MEYSU MGROS MHRGY MIATK MMCAS MNDRS MNDTR MOBTL MOGAN MOPAS MPARK MRGYO MRSHL
MSGYO MTRKS MTRYO MZHLD NATEN NETAS NETCD NETGL NIBAS NTGAZ NTHOL NUGYO NUHCM OBAMS OBASE ODAS
ODINE OFSYM ONCSM ONRYT ORCAY ORGE ORMA ORZAX OSMEN OSTIM OTKAR OTTO OYAKC OYAYO OYLUM OYYAT
OZATD OZGYO OZKGY OZRDN OZSUB OZYSR PAGYO PAHOL PAMEL PAPIL PARSN PASEU PATEK PCILT PEKGY PENGD
PENTA PETKM PETUN PGSUS PINSU PKART PKENT PLTUR PNLSN PNSUT POLHO POLTK PRDGS PRKAB PRKME PRZMA
PSDTC PSGYO QNBFK QNBTR QUAGR QUICK RALYH RAYSG REEDR RGYAS RNPOL RODRG RTALB RUBNS RUZYE RYGYO
RYSAS SAFKR SAHOL SAMAT SANEL SANFM SANKO SARAE SARKY SASA SAYAS SDTTR SEGMN SEGYO SEKFK SEKUR
SELEC SELVA SERNT SEYKM SILVR SISE SKBNK SKTAS SKYLP SKYMD SMART SMRTG SMRVA SNGYO SNICA SNPAM
SODSN SOHOE SOKE SOKM SONME SRVGY SSAAT SUMAS SUNTK SURGY SUWEN SVGYO TABGD TARKM TATEN TATGD
TAVHL TBORG TCELL TCKRC TDGYO TEHOL TEKTU TERA TEZOL TGSAS THYAO TKFEN TKNKA TKNSA TLMAN TMPOL
TMSN TNZTP TOASO TRALT TRCAS TRENJ TRGYO TRHOL TRILC TRMET TSGYO TSKB TSPOR TTKOM TTRAK TUCLK
TUKAS TUPRS TUREX TURGG TURSG UCAYM UFUK ULAS ULKER ULUFA ULUSE ULUUN UNLU USAK USHOL VAKBN
VAKFA VAKFN VAKKO VANGD VBTYZ VERUS VESBE VESTL VEYAS VKFYO VKGYO VKING VRGYO VSNMD YAPRK YATAS
YAYLA YBTAS YEOTK YESIL YGGYO YIGIT YKBNK YKSLN YONGA YUNSA YYAPI YYLGD ZEDUR ZERGY ZGYO ZOREN
ZRGYO
""".split())

def liste_olustur():
    for ad, fn in [('TradingView', tradingview_listesi), ('Is Yatirim', isyatirim_listesi)]:
        try:
            s = fn()
            print(f'  {ad}: {len(s)} hisse')
            if len(s) >= 400:
                return s, ad
            print(f'  {ad} listesi eksik gorunuyor, sonraki kaynaga geciliyor')
        except Exception as e:
            print(f'  {ad} hatasi: {e}')
    print('  Canli kaynaklara ulasilamadi, statik liste kullaniliyor')
    return set(STATIK), 'Statik'

print('Hisse listesi olusturuluyor...')
canli, kaynak = liste_olustur()
tum_tikerlar = sorted(canli - HARIC)

yeni   = sorted(set(tum_tikerlar) - STATIK)
cikmis = sorted(STATIK - set(tum_tikerlar) - HARIC)
print(f'\nKaynak: {kaynak} | Toplam: {len(tum_tikerlar)} hisse')
if kaynak != 'Statik':
    if yeni:   print(f'Statik listede olmayan yeni hisseler ({len(yeni)}):', ', '.join(yeni))
    if cikmis: print(f'Artik listede olmayanlar ({len(cikmis)}):', ', '.join(cikmis))

In [ ]:
#@title Veriyi TradingView'den indir

def _tv_mesaj(fonk, param):
    s = json.dumps({'m': fonk, 'p': param}, separators=(',', ':'))
    return f'~m~{len(s)}~m~{s}'

def tv_gunluk(sembol, duzeltme, bar=20000):
    """TradingView'den gunluk OHLC (giris yapmadan). Sembol bulunamazsa None."""
    ws = create_connection('wss://data.tradingview.com/socket.io/websocket',
                           header=['Origin: https://data.tradingview.com'], timeout=30)
    try:
        cs = 'cs_' + ''.join(random.choices(string.ascii_lowercase, k=12))
        ws.send(_tv_mesaj('set_auth_token', ['unauthorized_user_token']))
        ws.send(_tv_mesaj('chart_create_session', [cs, '']))
        tanim = json.dumps({'symbol': sembol, 'adjustment': duzeltme, 'session': 'regular'})
        ws.send(_tv_mesaj('resolve_symbol', [cs, 'sym', '=' + tanim]))
        ws.send(_tv_mesaj('create_series', [cs, 's1', 's1', 'sym', 'D', bar, '']))
        ham = ''
        while True:
            m = ws.recv()
            ham += m
            if re.fullmatch(r'~m~\d+~m~~h~\d+', m):          # heartbeat
                ws.send(m)
            if 'series_completed' in m:
                break
            if 'symbol_error' in m or 'series_error' in m:
                return None
    finally:
        ws.close()
    satir = [json.loads(x)['v'][:5] for x in re.findall(r'\{"i":\d+,"v":\[[^\]]+\]\}', ham)]
    if not satir:
        return None
    df = pd.DataFrame(satir, columns=['t', 'Open', 'High', 'Low', 'Close'])
    df.index = pd.to_datetime(df.pop('t'), unit='s').dt.normalize()
    df = df[~df.index.duplicated(keep='last')].sort_index()
    return df[(df[['Open', 'High', 'Low', 'Close']] > 0).all(axis=1)]

DUZELTME = 'dividends' if TEMETTU_DUZELTME else 'splits'

def indir(ticker, deneme=3):
    for i in range(deneme):
        try:
            return tv_gunluk(f'BIST:{ticker}/{ALTIN}', DUZELTME)
        except Exception:
            time.sleep(2 * (i + 1))
    return None

t0 = time.time()
veriler = {}
with ThreadPoolExecutor(ES_ZAMANLI) as ex:
    for i, (t, d) in enumerate(zip(tum_tikerlar, ex.map(indir, tum_tikerlar)), 1):
        if d is not None and len(d):
            veriler[t] = d
        if i % 50 == 0 or i == len(tum_tikerlar):
            print(f'  {i}/{len(tum_tikerlar)} indirildi ({len(veriler)} veri var) - {time.time()-t0:.0f} sn')

In [ ]:
#@title Tarama (ATL + ATH)

def bozuk_gunleri_at(d, kat=3):
    """Cevresindeki 11 gunun medyanindan 3 kat uzak tek gunluk hatali mumlari atar
    (ornek: BFREN 30.09.2004'te fiyat bir gunlugune 6000 kat dusmus gorunuyor)."""
    med = d['Close'].rolling(11, center=True, min_periods=3).median()
    bozuk = ((d['Close'] / med > kat) | (d['Close'] / med < 1 / kat) |
             (d['Low'] / med < 1 / kat) | (d['High'] / med > kat))
    return d[~bozuk], int(bozuk.sum())

dusuk_kol, yuksek_kol = ('Low', 'High') if FIYAT == 'fitil' else ('Close', 'Close')
sonuclar, hatalar, temizlenen = [], {}, {}
for ticker in tum_tikerlar:
    d = veriler.get(ticker)
    if d is None:
        hatalar[ticker] = 'veri yok'; continue
    d, n = bozuk_gunleri_at(d)
    if n:
        temizlenen[ticker] = n
    if len(d) < MIN_VERI_GUNU:
        hatalar[ticker] = f'az veri ({len(d)} gun)'; continue

    atl, ath, son = float(d[dusuk_kol].min()), float(d[yuksek_kol].max()), float(d['Close'].iloc[-1])
    sonuclar.append({
        'Hisse'       : ticker,
        'Son Tarih'   : f'{d.index[-1]:%Y-%m-%d}',
        'Son (XAU)'   : son,
        'ATL (XAU)'   : atl,
        'ATL Tarihi'  : f'{d[dusuk_kol].idxmin():%Y-%m-%d}',
        'ATL Uzaklik%': round((son - atl) / atl * 100, 2),   # ATL'den ne kadar yukarida
        'ATH (XAU)'   : ath,
        'ATH Tarihi'  : f'{d[yuksek_kol].idxmax():%Y-%m-%d}',
        'ATH Uzaklik%': round((son - ath) / ath * 100, 2),   # ATH'nin ne kadar altinda
        'Ilk Tarih'   : f'{d.index[0]:%Y-%m-%d}',
        'Gun'         : len(d),
    })

print(f'{len(sonuclar)} hisse islendi | {len(hatalar)} atlandi')
if hatalar:
    print('Atlananlar:', ', '.join(f'{k} ({v})' for k, v in hatalar.items()))
if temizlenen:
    print('Hatali mum ayiklananlar:', ', '.join(f'{k} ({v} gun)' for k, v in temizlenen.items()))

In [ ]:
#@title Sonuc tablolari
df = pd.DataFrame(sonuclar).sort_values('ATL Uzaklik%').reset_index(drop=True)
pd.set_option('display.float_format', lambda x: f'{x:.8f}' if 0 < abs(x) < 1 else f'{x:.2f}')
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 700)
pd.set_option('display.width', 200)

# Son verisi 7 gunden eski olanlar (islem durdurma vb.) uyari icin isaretlenir
son_gun = pd.to_datetime(df['Son Tarih']).max()
eski = df[pd.to_datetime(df['Son Tarih']) < son_gun - pd.Timedelta(days=7)]
if len(eski):
    print(f'Uyari - son verisi eski olan {len(eski)} hisse:', ', '.join(eski['Hisse']))

radar_df = df[df['ATL Uzaklik%'] <= ESIK_YUZDE].reset_index(drop=True)
print(f'\nRADAR - ATL uzaklik %{ESIK_YUZDE} veya daha az: {len(radar_df)} hisse')
print('='*80)
display(radar_df)
print('\nTUM LISTE (ATL yakinlik sirasi):')
print('='*80)
display(df)

In [ ]:
#@title Tek hisse kontrol
KONTROL = 'ENKAI'  #@param {type:"string"}
#@markdown Hissenin en dusuk / en yuksek 5 gununu gosterir (TradingView grafigiyle karsilastirmak icin)
k = KONTROL.strip().upper()
d = veriler.get(k)
if d is None:
    d = indir(k)
if d is not None:
    d, n = bozuk_gunleri_at(d)
    if n:
        print(f'{n} hatali mum ayiklandi')
    print(f'BIST:{k}/{ALTIN} ({DUZELTME}): {d.index[0]:%Y-%m-%d} - {d.index[-1]:%Y-%m-%d}, {len(d)} gun')
    print('\nEn dusuk 5 gun:'); display(d.nsmallest(5, dusuk_kol))
    print('\nEn yuksek 5 gun:'); display(d.nlargest(5, yuksek_kol))
    print('\nSon gun:'); display(d.tail(1))
else:
    print(f'{k} icin veri yok')

In [ ]:
#@title CSV kaydet / indir
DRIVE_KAYDET = True  #@param {type:"boolean"}
#@markdown Isaretliyse CSV Google Drive'da MyDrive/BIST_Tarama klasorune de kopyalanir (ilk seferde izin ister)
import os, shutil
from google.colab import files

dosya = f'bist_atl_ath_{datetime.now():%Y%m%d_%H%M}.csv'
df.to_csv(dosya, index=False, encoding='utf-8-sig', float_format='%.8f')
print(f'Kaydedildi: /content/{dosya}')
print('  -> Sol menudeki klasor simgesinden dosyaya sag tikla -> Indir')

if DRIVE_KAYDET:
    from google.colab import drive
    drive.mount('/content/drive')
    hedef = '/content/drive/MyDrive/BIST_Tarama'
    os.makedirs(hedef, exist_ok=True)
    shutil.copy(dosya, hedef)
    print(f'Drive: MyDrive/BIST_Tarama/{dosya}')

try:
    files.download(dosya)
    print('Tarayici indirmesi baslatildi (gelmezse tarayicinin indirme/acilir pencere iznini kontrol et)')
except Exception as e:
    print('Otomatik indirme calismadi:', e)